In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
import time
from typing import List, Dict, Optional
from dataclasses import dataclass
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urljoin, urlparse
import re

In [ ]:

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


@dataclass
class Article:
    """Data class for article information"""
    title: str
    content: str
    url: str
    source: str
    scores: Optional[Dict[str, float]] = None
    
    def to_dict(self):
        return {
            'title': self.title,
            'content': self.content[:500],  # Preview only
            'url': self.url,
            'source': self.source,
            **({f'score_{k}': v for k, v in self.scores.items()} if self.scores else {})
        }


class ArticleScraper:
    """
    Flexible web scraper for news and research articles
    """
    
    def __init__(self, timeout=10, max_retries=3):
        self.timeout = timeout
        self.max_retries = max_retries
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
    
    def scrape_url(self, url: str) -> Optional[Article]:
        """
        Scrape a single article URL
        
        Returns Article object or None if scraping fails
        """
        for attempt in range(self.max_retries):
            try:
                response = requests.get(url, headers=self.headers, timeout=self.timeout)
                response.raise_for_status()
                
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # Extract title and content using multiple strategies
                title = self._extract_title(soup)
                content = self._extract_content(soup)
                
                if not title or not content:
                    logger.warning(f"Could not extract title or content from {url}")
                    return None
                
                source = urlparse(url).netloc
                
                return Article(
                    title=title,
                    content=content,
                    url=url,
                    source=source
                )
                
            except requests.RequestException as e:
                logger.error(f"Attempt {attempt + 1}/{self.max_retries} failed for {url}: {e}")
                if attempt < self.max_retries - 1:
                    time.sleep(2 ** attempt)  # Exponential backoff
                continue
        
        return None
    
    def _extract_title(self, soup: BeautifulSoup) -> str:
        """Extract article title using multiple strategies"""
        # Strategy 1: OpenGraph meta tag
        og_title = soup.find('meta', property='og:title')
        if og_title and og_title.get('content'):
            return og_title['content'].strip()
        
        # Strategy 2: Standard title tag
        if soup.title:
            return soup.title.string.strip()
        
        # Strategy 3: h1 tag
        h1 = soup.find('h1')
        if h1:
            return h1.get_text(strip=True)
        
        return ""
    
    def _extract_content(self, soup: BeautifulSoup) -> str:
        """Extract article content using multiple strategies"""
        # Strategy 1: OpenGraph description
        og_desc = soup.find('meta', property='og:description')
        if og_desc and og_desc.get('content'):
            content = og_desc['content'].strip()
        else:
            content = ""
        
        # Strategy 2: Article tag
        article = soup.find('article')
        if article:
            paragraphs = article.find_all('p')
            content += ' ' + ' '.join([p.get_text(strip=True) for p in paragraphs])
        
        # Strategy 3: Main content div (common patterns)
        if not content:
            main_content = soup.find(['div'], class_=re.compile(r'(article|content|post|entry|body)', re.I))
            if main_content:
                paragraphs = main_content.find_all('p')
                content = ' '.join([p.get_text(strip=True) for p in paragraphs])
        
        # Strategy 4: All paragraphs as fallback
        if not content or len(content) < 100:
            paragraphs = soup.find_all('p')
            content = ' '.join([p.get_text(strip=True) for p in paragraphs[:10]])
        
        # Clean content
        content = re.sub(r'\s+', ' ', content).strip()
        
        return content
    
    def scrape_arxiv(self, arxiv_id: str) -> Optional[Article]:
        """
        Specialized scraper for arXiv research papers
        
        Args:
            arxiv_id: arXiv ID (e.g., "2301.12345")
        """
        url = f"https://arxiv.org/abs/{arxiv_id}"
        
        try:
            response = requests.get(url, headers=self.headers, timeout=self.timeout)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Title
            title_tag = soup.find('h1', class_='title')
            title = title_tag.get_text(strip=True).replace('Title:', '').strip() if title_tag else ""
            
            # Abstract
            abstract_tag = soup.find('blockquote', class_='abstract')
            abstract = abstract_tag.get_text(strip=True).replace('Abstract:', '').strip() if abstract_tag else ""
            
            return Article(
                title=title,
                content=abstract,
                url=url,
                source='arxiv.org'
            )
            
        except Exception as e:
            logger.error(f"Failed to scrape arXiv {arxiv_id}: {e}")
            return None
    
    def scrape_multiple(self, urls: List[str], max_workers=5) -> List[Article]:
        """
        Scrape multiple URLs concurrently
        
        Args:
            urls: List of article URLs
            max_workers: Number of concurrent threads
        """
        articles = []
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_url = {executor.submit(self.scrape_url, url): url for url in urls}
            
            for future in as_completed(future_to_url):
                url = future_to_url[future]
                try:
                    article = future.result()
                    if article:
                        articles.append(article)
                        logger.info(f"Successfully scraped: {article.title[:50]}...")
                except Exception as e:
                    logger.error(f"Error processing {url}: {e}")
        
        return articles

In [ ]:
class MultiDimensionalAnalyzer:
    """
    Analyzes articles across 5 dimensions
    """
    
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.embedding_model = SentenceTransformer(model_name)
        self.regression_model = MultiOutputRegressor(
            RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
        )
        self.is_fitted = False
        
        self.dimensions = [
            'growth_potential',
            'recession_resistance',
            'automation_resistance',
            'skill_accessibility',
            'cross_industry_collaboration'
        ]
    
    def train(self, texts: List[str], scores_df: pd.DataFrame):
        """Train the model on labeled data"""
        logger.info("Creating embeddings for training data...")
        embeddings = self.embedding_model.encode(texts, show_progress_bar=True, batch_size=32)
        
        scores_array = scores_df[self.dimensions].values
        
        logger.info("Training multi-output model...")
        self.regression_model.fit(embeddings, scores_array)
        self.is_fitted = True
        
        logger.info("Training complete!")
        return self
    
    def score_articles(self, articles: List[Article]) -> List[Article]:
        """
        Score multiple articles across all dimensions
        
        Args:
            articles: List of Article objects
            
        Returns:
            Same articles with scores populated
        """
        if not self.is_fitted:
            raise ValueError("Model must be trained first!")
        
        # Combine title and content for better context
        texts = [f"{art.title}. {art.content}" for art in articles]
        
        logger.info(f"Scoring {len(articles)} articles...")
        embeddings = self.embedding_model.encode(texts, show_progress_bar=True, batch_size=32)
        predictions = self.regression_model.predict(embeddings)
        predictions = np.clip(predictions, 0, 10)
        
        # Add scores to articles
        for i, article in enumerate(articles):
            article.scores = {
                dim: float(predictions[i, j]) 
                for j, dim in enumerate(self.dimensions)
            }
        
        return articles
    def visualize_predictions(self, text, predictions=None):
        """Create radar chart for a single field's scores"""
        if predictions is None:
            predictions = self.predict_single(text)
        
        # Prepare data for radar chart
        categories = [dim.replace('_', ' ').title() for dim in self.dimensions]
        values = [predictions[dim] for dim in self.dimensions]
        
        # Number of variables
        N = len(categories)
        
        # Compute angle for each axis
        angles = [n / float(N) * 2 * np.pi for n in range(N)]
        values += values[:1]  # Complete the circle
        angles += angles[:1]
        
        # Create plot
        fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))
        
        ax.plot(angles, values, 'o-', linewidth=2, color='#3B82F6')
        ax.fill(angles, values, alpha=0.25, color='#3B82F6')
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(categories, size=10)
        ax.set_ylim(0, 10)
        ax.set_yticks([2, 4, 6, 8, 10])
        ax.grid(True)
        
        plt.title("Field Analysis Profile", size=14, pad=20)
        plt.tight_layout()
        
        return fig
    
    def compare_fields(self, texts_dict):
        """
        Compare multiple fields
        
        Parameters:
        - texts_dict: {'Field Name': 'description text', ...}
        
        Returns: DataFrame with all predictions
        """
        texts = list(texts_dict.values())
        names = list(texts_dict.keys())
        
        predictions = self.predict(texts)
        predictions.insert(0, 'Field', names)
        
        return predictions
    
    def get_top_fields(self, predictions_df, dimension, n=10):
        """Get top N fields for a specific dimension"""
        return predictions_df.nlargest(n, dimension)[['Field', dimension]]

In [ ]:
class ArticlePipeline:
    """
    Complete pipeline: Scrape -> Analyze -> Store
    """
    
    def __init__(self, analyzer: MultiDimensionalAnalyzer, scraper: Optional[ArticleScraper] = None):
        self.analyzer = analyzer
        self.scraper = scraper or ArticleScraper()
        self.vectorizedArticles: List[Dict] = []  # Main storage
    
    def process_urls(self, urls: List[str], max_workers=5) -> pd.DataFrame:
        """
        Complete pipeline: scrape URLs, analyze, and store
        
        Args:
            urls: List of article URLs to scrape
            max_workers: Concurrent scraping threads
            
        Returns:
            DataFrame with results
        """
        logger.info(f"Starting pipeline for {len(urls)} URLs...")
        
        # Step 1: Scrape articles
        articles = self.scraper.scrape_multiple(urls, max_workers=max_workers)
        logger.info(f"Successfully scraped {len(articles)}/{len(urls)} articles")
        
        if not articles:
            logger.warning("No articles scraped successfully!")
            return pd.DataFrame()
        
        # Step 2: Score articles
        scored_articles = self.analyzer.score_articles(articles)
        
        # Step 3: Store in vectorizedArticles
        for article in scored_articles:
            self.vectorizedArticles.append({
                'title': article.title,
                'url': article.url,
                'source': article.source,
                **article.scores
            })
        
        # Return as DataFrame for easy viewing
        df = pd.DataFrame(self.vectorizedArticles)
        logger.info(f"Pipeline complete! Total articles in vectorizedArticles: {len(self.vectorizedArticles)}")
        
        return df
    
    def process_arxiv_papers(self, arxiv_ids: List[str]) -> pd.DataFrame:
        """Process arXiv papers specifically"""
        logger.info(f"Processing {len(arxiv_ids)} arXiv papers...")
        
        articles = []
        for arxiv_id in arxiv_ids:
            article = self.scraper.scrape_arxiv(arxiv_id)
            if article:
                articles.append(article)
            time.sleep(1)  # Respect rate limits
        
        logger.info(f"Successfully scraped {len(articles)}/{len(arxiv_ids)} papers")
        
        if articles:
            scored_articles = self.analyzer.score_articles(articles)
            
            for article in scored_articles:
                self.vectorizedArticles.append({
                    'title': article.title,
                    'url': article.url,
                    'source': article.source,
                    **article.scores
                })
        
        return pd.DataFrame(self.vectorizedArticles)
    
    def export_results(self, filename='article_scores.csv'):
        """Export vectorizedArticles to CSV"""
        df = pd.DataFrame(self.vectorizedArticles)
        df.to_csv(filename, index=False)
        logger.info(f"Exported {len(self.vectorizedArticles)} articles to {filename}")
        return df
    
    def get_top_articles(self, dimension: str, n=10) -> pd.DataFrame:
        """Get top N articles for a specific dimension"""
        df = pd.DataFrame(self.vectorizedArticles)
        return df.nlargest(n, dimension)[['title', 'source', dimension]]

In [ ]:
# ============================================================================
# EXAMPLE USAGE
# ============================================================================

if __name__ == "__main__":
    # Step 1: Train the analyzer on your labeled data
    print("=" * 80)
    print("STEP 1: Training the analyzer")
    print("=" * 80)
    
    # Load your labeled training data
    labeled_df = pd.read_csv('labeled_fields.csv')  # Your labeled dataset
    
    analyzer = MultiDimensionalAnalyzer()
    analyzer.train(
        texts=labeled_df['text'].tolist(),
        scores_df=labeled_df
    )
    
    # Step 2: Initialize the pipeline
    print("\n" + "=" * 80)
    print("STEP 2: Initializing pipeline")
    print("=" * 80)
    
    pipeline = ArticlePipeline(analyzer=analyzer)
    
    # Step 3: Scrape and analyze articles
    print("\n" + "=" * 80)
    print("STEP 3: Scraping and analyzing articles")
    print("=" * 80)
    
    # Example URLs (replace with your target sources)
    news_urls = [
        'https://www.nature.com/articles/d41586-024-00001-x',
        'https://techcrunch.com/2024/01/15/ai-breakthrough/',
        'https://www.theverge.com/23950084/quantum-computing-breakthrough',
        # Add more URLs here
    ]
    
    # Process news articles
    results_df = pipeline.process_urls(news_urls, max_workers=5)
    print("\n" + results_df.to_string())
    
    # Process arXiv papers
    arxiv_ids = ['2401.12345', '2401.67890']  # Example arXiv IDs
    arxiv_df = pipeline.process_arxiv_papers(arxiv_ids)
    
    # Step 4: Access vectorizedArticles
    print("\n" + "=" * 80)
    print("STEP 4: Accessing vectorizedArticles")
    print("=" * 80)
    
    print(f"\nTotal articles processed: {len(pipeline.vectorizedArticles)}")
    print("\nFirst 3 articles in vectorizedArticles:")
    for i, article in enumerate(pipeline.vectorizedArticles[:3]):
        print(f"\n{i+1}. {article['title']}")
        print(f"   Growth: {article['growth_potential']:.1f}, "
              f"Recession: {article['recession_resistance']:.1f}, "
              f"Automation: {article['automation_resistance']:.1f}")
    
    # Step 5: Analysis and export
    print("\n" + "=" * 80)
    print("STEP 5: Analysis and export")
    print("=" * 80)
    
    # Get top articles for specific dimension
    print("\nTop 5 articles by Growth Potential:")
    print(pipeline.get_top_articles('growth_potential', n=5))
    
    # Export results
    pipeline.export_results('article_analysis_results.csv')
    
    # Access the vectorizedArticles variable directly
    vectorizedArticles = pipeline.vectorizedArticles
    print(f"\nvectorizedArticles contains {len(vectorizedArticles)} articles")